# Group XX - Machine Learning, Element 1: Apply dataset preparation mechanisms
| Item | Details |
|---|---|
| Group | **Group XX** |
| Members | **IGIHOZO Bienvenue** and **MUGISHA Aime Patrick** |
| Question | **Q2 - Next-day rain early warning for MINEMA district disaster offices (own scenario)** |
| Stage | **4 - Clean and pre-process the data** |
| Driver / Navigator | Driver: IGIHOZO Bienvenue / Navigator: MUGISHA Aime Patrick |


## Stage 4 - Clean and pre-process the data (PC 1.3)
Each step calls a documented function in `src/rain_pipeline.py`; every decision goes into the Cleaning Log.

In [1]:
import sys  # lets us tell Python where our own code (src/) is
from pathlib import Path  # file paths that work on every computer
import warnings  # hide harmless library notices
warnings.filterwarnings("ignore")  # keep output readable
import numpy as np  # maths on whole columns
import pandas as pd  # DataFrame tables
ROOT = Path.cwd().parent  # notebooks/ is inside q2_own_scenario
sys.path.insert(0, str(ROOT / "src"))  # make src/rain_pipeline.py importable
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)  # show all columns
RAW_DIR = ROOT / "data" / "raw" / "gsod"  # original NOAA files (never edited)
import rain_pipeline as rp  # our reusable functions

raw = rp.load_raw(RAW_DIR)  # original data
log = rp.CleaningLog()  # empty log
def show(before, after):  # before/after helper
    print(f"shape {before.shape} -> {after.shape}")  # size change
    common = [c for c in after.columns if c in before.columns]  # columns in both
    print("missing (after):", after.isna().sum()[lambda s: s > 0].to_dict())  # remaining missing values

### Step 1: trim padded text, convert types

In [2]:
df = rp.trim_and_type(raw, log)  # strip spaces, text -> numbers/dates
print(raw.dtypes.value_counts().to_dict(), "->", df.dtypes.astype(str).value_counts().to_dict())  # types before/after

{<StringDtype(storage='python', na_value=nan)>: 29} -> {'Float64': 15, 'string': 13, 'datetime64[us]': 1}


### Step 2: missing codes -> NaN

In [3]:
step = df.copy()  # before
df = rp.replace_missing_codes(df, log)  # 9999.9 etc. -> NaN
pd.DataFrame({"max_before": step[list(rp.MISSING_CODES)].max(), "max_after": df[list(rp.MISSING_CODES)].max(),
              "missing_after": df[list(rp.MISSING_CODES)].isna().sum()})  # codes disappear, NaN appear

,max_before,max_after,missing_after
TEMP,83.1,83.1,0
DEWP,9999.9,67.9,26
SLP,9999.9,<NA>,10614
STP,999.9,885.2,8997
VISIB,999.9,7.8,30
WDSP,999.9,32.1,85
MXSPD,999.9,60.0,1212
GUST,999.9,50.5,10165
MAX,9999.9,93.2,2
MIN,9999.9,80.6,2


### Step 3: drop empty columns

In [4]:
step = df.copy()  # before
df = rp.drop_empty_columns(df, log)  # SLP, SNDP, GUST, STP + bookkeeping columns
print("removed:", sorted(set(step.columns) - set(df.columns)))  # which columns went
show(step, df)  # before/after

removed: ['DEWP_ATTRIBUTES', 'GUST', 'LATITUDE', 'LONGITUDE', 'MAX_ATTRIBUTES', 'MIN_ATTRIBUTES', 'SLP', 'SLP_ATTRIBUTES', 'SNDP', 'STP', 'STP_ATTRIBUTES', 'VISIB_ATTRIBUTES', 'WDSP_ATTRIBUTES']
shape (10614, 29) -> (10614, 16)
missing (after): {'DEWP': 26, 'VISIB': 30, 'WDSP': 85, 'MXSPD': 1212, 'MAX': 2, 'MIN': 2, 'PRCP': 862}


### Step 4-5: station names, metric units

In [5]:
step = df.copy()  # before
df = rp.name_stations(df, log)  # short name, district, province
df = rp.convert_units(df, log)  # F -> C, knots -> km/h, miles -> km, inches -> mm
print(df[["station", "district", "province"]].drop_duplicates().to_string(index=False))  # station table
df[["station", "DATE", "temp_c", "dewpoint_c", "max_temp_c", "min_temp_c", "wind_kmh", "visibility_km", "prcp_mm"]].describe().round(1)  # metric ranges

  station district    province
  Kamembe   Rusizi     Western
  Gisenyi   Rubavu     Western
Ruhengeri  Musanze    Northern
   Kigali Kicukiro Kigali City


,DATE,temp_c,dewpoint_c,max_temp_c,min_temp_c,wind_kmh,visibility_km,prcp_mm
count,10614,10614.0,10588.0,10612.0,10612.0,10529.0,10584.0,9752.0
mean,2020-01-06 17:46:21.910684,21.5,15.5,26.3,16.1,5.8,9.7,3.4
min,2015-01-01 00:00:00,14.5,4.2,16.0,5.8,0.0,3.9,0.0
25%,2017-11-26 00:00:00,20.5,14.5,25.0,15.0,2.8,9.6,0.0
50%,2019-11-12 00:00:00,21.6,15.9,26.1,16.4,6.5,10.0,0.0
75%,2021-11-25 00:00:00,22.7,16.9,27.5,17.0,8.3,10.0,2.0
max,2025-08-24 00:00:00,28.4,19.9,34.0,27.0,59.4,12.6,490.0
std,NaN,1.6,2.1,1.9,2.2,3.7,0.7,12.1


### Step 6-7: decode flags, impossible values

In [6]:
step = df.copy()  # before
df = rp.quality_flags(df, log)  # FRSHTT -> fog/rain/hail/thunder; PRCP flag; n_obs
df = rp.impossible_values(df, log)  # impossible combinations / unrealistic wind -> NaN
print(df[["fog", "rain", "hail", "thunder", "prcp_incomplete"]].mean().round(3).to_dict())  # share of days with each event
print("max wind before/after:", step["max_wind_kmh"].max(), "->", df["max_wind_kmh"].max())  # effect

{'fog': 0.043, 'rain': 0.386, 'hail': 0.002, 'thunder': 0.342, 'prcp_incomplete': 0.361}
max wind before/after: 111.12 -> 82.78


### Step 8-9: duplicates, target `rain_tomorrow`

In [7]:
step = df.copy()  # before
df = rp.remove_duplicates(df, log)  # one row per station-day
df = rp.add_target(df, log)  # rain flag of the NEXT calendar day; rows without a real tomorrow removed
show(step, df)  # before/after
print("rain_tomorrow share:", round(df[rp.TARGET].mean(), 3))  # balance

shape (10614, 20) -> (9818, 22)
missing (after): {'dewpoint_c': 18, 'max_temp_c': 1, 'min_temp_c': 1, 'wind_kmh': 71, 'max_wind_kmh': 1064, 'visibility_km': 26, 'prcp_mm': 812, 'days_since_prev_record': 2}
rain_tomorrow share: 0.392


### Reproducibility: one call gives the same table

In [8]:
again, _ = rp.clean_rain(rp.load_raw(RAW_DIR))  # all steps at once
pd.testing.assert_frame_equal(df, again)  # error if anything differs
print("identical:", again.shape)  # confirmation

identical: (9818, 22)


In [9]:
clean_log = log.to_frame()  # the log
clean_log.to_csv(ROOT / "docs" / "cleaning_log.csv", index=False)  # save
(ROOT / "data" / "interim").mkdir(parents=True, exist_ok=True)  # folder
df.to_csv(ROOT / "data" / "interim" / "rain_clean.csv", index=False)  # clean dataset
clean_log[["step", "operation", "column", "problem", "action", "records_affected"]]  # display

,step,operation,column,problem,action,records_affected
0,1,trim_and_type,all,values padded with spaces (' 70.0'),spaces removed,128312
1,2,trim_and_type,"TEMP, DEWP, SLP, STP, VISIB, WDSP, MXSPD, GUST...",numbers and dates stored as text,converted to float / datetime,10614
2,3,missing_codes,TEMP,missing code 9999.9,set to NaN,0
3,4,missing_codes,DEWP,missing code 9999.9,set to NaN,26
4,5,missing_codes,SLP,missing code 9999.9,set to NaN,10614
5,6,missing_codes,STP,missing code 999.9,set to NaN,8997
6,7,missing_codes,VISIB,missing code 999.9,set to NaN,30
7,8,missing_codes,WDSP,missing code 999.9,set to NaN,85
8,9,missing_codes,MXSPD,missing code 999.9,set to NaN,1212
9,10,missing_codes,GUST,missing code 999.9,set to NaN,10165


### Chronological split (train 2015-2022, validation 2023, test 2024-2025)
Weather is a time series: a **random** split would let the model learn from days after the test days (leakage).
We train on the past and test on the most recent years - exactly how MINEMA would use the model.

In [10]:
train, val, test = rp.chronological_split(df)  # split by date
for n, part in [("train", train), ("val", val), ("test", test)]:  # each part
    print(f"{n:<5} {part.shape} {part['DATE'].min().date()} -> {part['DATE'].max().date()}  rain share {part[rp.TARGET].mean():.3f}  stations {sorted(part.station.unique())}")
    part.to_csv(ROOT / "data" / "interim" / f"{n}_clean.csv", index=False)  # save un-encoded split

train (8438, 22) 2015-01-01 -> 2022-12-31  rain share 0.387  stations ['Gisenyi', 'Kamembe', 'Kigali', 'Ruhengeri']
val   (641, 22) 2023-01-01 -> 2023-12-31  rain share 0.413  stations ['Kamembe', 'Kigali']
test  (739, 22) 2024-01-01 -> 2025-08-23  rain share 0.441  stations ['Gisenyi', 'Kamembe', 'Kigali', 'Ruhengeri']


**Notes:** the validation year 2023 has only Kigali and Kamembe (NOAA has no 2023 data for Gisenyi and Ruhengeri) - a known
limitation. The split is about 86 / 7 / 7 % instead of 70/15/15 because a time split must follow the calendar; it still
gives 640+ validation and 730+ test days. The target is moderately imbalanced (~39 % rain) -> planned remedy:
`class_weight="balanced"`.

### Imputation, encoding, scaling - fitted on the training set only

In [11]:
from sklearn.compose import ColumnTransformer  # column-wise recipe
from sklearn.pipeline import Pipeline  # ordered steps
from sklearn.impute import SimpleImputer  # missing values
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # encoding and scaling
num_cols = ["temp_c", "dewpoint_c", "max_temp_c", "min_temp_c", "wind_kmh", "max_wind_kmh", "visibility_km", "prcp_mm", "n_obs"]  # numbers
flag_cols = ["fog", "rain", "hail", "thunder", "prcp_incomplete"]  # 0/1 flags (no scaling needed)
cat_cols = ["station"]  # category
prep = ColumnTransformer([  # recipe
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),  # median + z-score
    ("flag", SimpleImputer(strategy="most_frequent"), flag_cols),  # flags kept as 0/1
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),  # one column per station
])
prep.fit(train)  # LEARN from training rows only
Xtr, Xva, Xte = (prep.transform(p) for p in (train, val, test))  # apply to all
print("train", Xtr.shape, "val", Xva.shape, "test", Xte.shape)  # same columns
pd.DataFrame(Xtr, columns=prep.get_feature_names_out()).head().round(2)  # first encoded rows

train (8438, 18) val (641, 18) test (739, 18)


,num__temp_c,num__dewpoint_c,num__max_temp_c,num__min_temp_c,num__wind_kmh,num__max_wind_kmh,num__visibility_km,num__prcp_mm,num__n_obs,flag__fog,flag__rain,flag__hail,flag__thunder,flag__prcp_incomplete,cat__station_Gisenyi,cat__station_Kamembe,cat__station_Kigali,cat__station_Ruhengeri
0,0.61,0.78,0.40,-0.13,0.53,-0.31,0.34,-0.03,-0.99,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,-0.03,1.17,1.04,0.52,0.79,-0.31,-1.12,-0.27,-0.99,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,0.90,0.78,0.92,-0.52,-0.25,-1.03,-0.63,1.66,-0.53,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,1.79,0.92,0.40,0.42,0.32,-0.56,0.34,-0.27,-0.53,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1.82,0.69,0.92,0.42,0.12,-0.81,0.34,-0.27,-0.38,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
